# 3 · Splitting

A model is trained on one part of the data and evaluated on another. This guide
shows how WarpRec makes those parts from a single file:

- the eight splitting strategies, the parameters each reads, and a check on
  MovieLens-100K that each split has the property it promises;
- alignment: what happens to test rows whose user or item training never saw;
- seeds, and what else a split depends on;
- a validation split on top of the test split, as a holdout or as k folds, and
  which rows each resulting dataset trains and evaluates on.

Everything is done through the `Splitter` API, then through the `splitter`
section of a configuration file.

**Requirements:** `pip install warprec jupyter`. Runs in under a minute; it
downloads MovieLens-100K (5 MB) into `guides/data/` if guide 1 has not.

In [1]:
import sys
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import polars as pl
from loguru import logger

sys.path.insert(0, "..")
from guide_data import movielens_100k

logger.disable("warprec")

`movielens_100k()` downloads the dataset and writes `ratings.tsv`, exactly as
guide 1 does step by step. The reader returns a Narwhals frame over Polars; the
checks below work on the Polars frames underneath.

In [2]:
from warprec.data.reader import LocalReader

ratings_path = movielens_100k() / "ratings.tsv"
data = LocalReader().read_tabular(local_path=str(ratings_path), sep="\t")
ratings = data.to_native()
print(
    f"{len(ratings):,} interactions, {ratings['user_id'].n_unique()} users, "
    f"{ratings['item_id'].n_unique()} items"
)

100,000 interactions, 943 users, 1682 items


## The API

`Splitter().split_transaction` takes the data and one `SplitSpec` per split it
should make: `test=` always, `validation=` optionally. It returns
`(train, validation, test)`, with `validation` set to `None` when none was asked
for.

A `SplitSpec` holds the strategy and the parameters it reads: `ratio`, `k`,
`folds`, `timestamp` and `seed` (42 unless given).

| strategy | reads | holds out |
|---|---|---|
| `temporal_holdout` | `ratio` | the latest `ratio` of each user's interactions |
| `random_holdout` | `ratio`, `seed` | a random `ratio` of all interactions, regardless of user |
| `temporal_leave_k_out` | `k` | each user's `k` latest interactions |
| `random_leave_k_out` | `k`, `seed` | `k` random interactions per user |
| `timestamp_slicing` | `timestamp` (an integer or `"best"`) | every interaction at or after the timestamp |
| `k_fold_cross_validation` | `folds` | validation only: each user's interactions dealt into `folds` folds |
| `item_cold_start` | `ratio`, `seed` | every interaction of a random `ratio` of the **items** |
| `user_cold_start` | `ratio`, `seed` | every interaction of a random `ratio` of the **users** |

Two things to know about the API. Leaving `test=` out does not mean "no test
split": the call fails with an `AttributeError`, so always pass it. And a
`SplitSpec` does not check that the parameters its strategy needs are set
(the configuration file does): a `temporal_holdout` without `ratio` fails with
a `TypeError` from inside the strategy.

In [3]:
from warprec.data.schema import SplitSpec
from warprec.data.splitting import Splitter

splitter = Splitter()
train, validation, test = splitter.split_transaction(
    data, test=SplitSpec(strategy="temporal_holdout", ratio=0.1)
)
print(type(train).__name__, train.shape, validation, test.shape)

DataFrame (89561, 4) None (10401, 4)


## Every test strategy on MovieLens-100K

The cell below runs the seven strategies that can make a test set (k-fold is
validation only), with the parameters a paper would typically use. The fixed
pivot of `timestamp_slicing` is 1 March 1998, about five weeks before the
dataset's last rating.

`process_split` runs the strategy alone; `split_transaction` adds the alignment
described in the next section. Both are kept so the table can show what
alignment removed.

In [4]:
PIVOT = int(datetime(1998, 3, 1, tzinfo=timezone.utc).timestamp())

specs = {
    "temporal_holdout": SplitSpec(strategy="temporal_holdout", ratio=0.1),
    "random_holdout": SplitSpec(strategy="random_holdout", ratio=0.1),
    "temporal_leave_k_out": SplitSpec(strategy="temporal_leave_k_out", k=1),
    "random_leave_k_out": SplitSpec(strategy="random_leave_k_out", k=1),
    "timestamp_slicing (fixed)": SplitSpec(
        strategy="timestamp_slicing", timestamp=PIVOT
    ),
    "timestamp_slicing (best)": SplitSpec(
        strategy="timestamp_slicing", timestamp="best"
    ),
    "item_cold_start": SplitSpec(strategy="item_cold_start", ratio=0.1),
    "user_cold_start": SplitSpec(strategy="user_cold_start", ratio=0.1),
}


def run(spec: SplitSpec) -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """Split the ratings; return train, aligned test and the test before alignment."""
    _, raw_test = splitter.process_split(data, spec)[0]
    train, _, test = splitter.split_transaction(data, test=spec)
    return train.to_native(), test.to_native(), raw_test.to_native()


splits = {name: run(spec) for name, spec in specs.items()}
pd.DataFrame(
    {
        name: {
            "train rows": len(train),
            "test rows": len(test),
            "dropped by alignment": len(raw_test) - len(test),
            "train users": train["user_id"].n_unique(),
            "test users": test["user_id"].n_unique(),
            "train items": train["item_id"].n_unique(),
            "test items": test["item_id"].n_unique(),
        }
        for name, (train, test, raw_test) in splits.items()
    }
).T

,train rows,test rows,dropped by alignment,train users,test users,train items,test items
temporal_holdout,89561,10401,38,943,943,1648,1335
random_holdout,90000,9985,15,943,923,1667,1216
temporal_leave_k_out,99057,941,2,943,941,1680,527
random_leave_k_out,99057,942,1,943,942,1681,486
timestamp_slicing (fixed),77985,3715,18300,736,120,1608,1026
timestamp_slicing (best),59968,5243,34789,590,141,1511,1069
item_cold_start,89708,10292,0,943,931,1514,168
user_cold_start,90915,9053,32,849,94,1651,1282


A few things the table already shows:

- `temporal_holdout` and `random_holdout` both hold out 10% of the rows, but
  only the temporal one does it per user: every user has test rows there, while
  the random one leaves some users without any.
- The leave-one-out splits hold out exactly 943 rows before alignment, one per
  user.
- The cold-start splits hold out entities, not rows: 10% of the 1,682 items is
  168 items, and their interactions are 10,292 rows.
- `timestamp_slicing` keeps far fewer test users than the other strategies,
  and alignment removes most of its test rows.

## Alignment

A model can only score users and items it has indices for, and only the
training split defines indices (guide 1). So after splitting, `split_transaction`
drops every test row whose user or whose item has no row in the training set,
and logs how many it dropped. With the log enabled, the fixed slicing shows it:

In [5]:
logger.enable("warprec")
_ = splitter.split_transaction(data, test=specs["timestamp_slicing (fixed)"])
logger.disable("warprec")

08-10-2026 12:22:26 - WarpRec - 📝 Starting test splitting process with timestamp_slicing splitting strategy.


08-10-2026 12:22:26 - WarpRec - 📝 Test splitting completed in : 0.00s


08-10-2026 12:22:26 - WarpRec - ✅ Splitting process over in 0.00s.


08-10-2026 12:22:26 - WarpRec - 📢 Test set was not aligned with the training set. Filtered out 18300 transactions.


A global pivot puts every user who joined MovieLens after it entirely on the
test side; none of their rows can be evaluated. The per-user strategies keep
every user on both sides, so alignment there only removes the few rows of items
that happen to appear only in the test set (38 for `temporal_holdout`).

The cold-start strategies are the exception: the side they hold out is unseen
on purpose, so it is not filtered. `item_cold_start` keeps every test row
(all users have training rows); `user_cold_start` keeps its unseen users and
still drops the rows of items training never saw. Evaluating them is the topic
of guide 13.

## Checking each strategy

### Holdout and leave-k-out

A temporal strategy promises that, for every user, nothing in the test set is
older than anything in the training set. The table counts the users for whom
that fails, the number of test rows per user, and the users left with no test
row after alignment.

In [6]:
def per_user_check(train: pl.DataFrame, test: pl.DataFrame) -> dict:
    """Compare each user's newest training row with their oldest test row."""
    bounds = (
        test.group_by("user_id")
        .agg(first_test=pl.col("timestamp").min(), n_test=pl.len())
        .join(
            train.group_by("user_id").agg(last_train=pl.col("timestamp").max()),
            on="user_id",
        )
    )
    return {
        "users with a test row older than a train row": (
            bounds["first_test"] < bounds["last_train"]
        ).sum(),
        "test rows per user (min)": bounds["n_test"].min(),
        "test rows per user (max)": bounds["n_test"].max(),
        "users without test rows": ratings["user_id"].n_unique() - len(bounds),
    }


pd.DataFrame(
    {
        name: per_user_check(*splits[name][:2])
        for name in [
            "temporal_holdout",
            "random_holdout",
            "temporal_leave_k_out",
            "random_leave_k_out",
        ]
    }
).T

,users with a test row older than a train row,test rows per user (min),test rows per user (max),users without test rows
temporal_holdout,0,2,67,0
random_holdout,916,1,78,20
temporal_leave_k_out,0,1,1,2
random_leave_k_out,892,1,1,1


- The temporal strategies never put a test row before a training row of the
  same user (a tie on the timestamp is broken by file order). The random ones
  do, for most users: they are not meant for sequential models.
- `temporal_holdout` keeps the oldest `⌊n·(1−ratio)⌋` rows of a user with `n`
  interactions, and always at least the first one, so a user with 20 ratings
  gets 2 test rows. `random_holdout` draws over all rows, so the count per user
  varies and 20 users get none.
- Leave-one-out gives every user exactly one test row. Two users (one in the random
  split) have none left in the table: alignment dropped their only test row because its item
  appears nowhere in training.

The leave-k-out strategies need a user to keep at least one training row, so a
user with `k` or fewer interactions is **removed from both sides**, not kept in
training. Every MovieLens-100K user has at least 20 ratings, so `k=1` removes
nobody; `k=20` removes the users with exactly 20:

In [7]:
counts = ratings.group_by("user_id").len()
short = counts.filter(pl.col("len") <= 20)["user_id"].to_list()
train, _, test = run(SplitSpec(strategy="temporal_leave_k_out", k=20))
print(
    f"{len(short)} users have 20 or fewer ratings; in the k=20 split they own "
    f"{train['user_id'].is_in(short).sum()} train rows and "
    f"{test['user_id'].is_in(short).sum()} test rows"
)
print(f"{len(train) + len(test):,} of {len(ratings):,} rows remain before alignment")

32 users have 20 or fewer ratings; in the k=20 split they own 0 train rows and 0 test rows
99,360 of 100,000 rows remain before alignment


### Timestamp slicing

Slicing promises a global order instead: every training row is older than every
test row. With `timestamp: best` WarpRec picks the pivot itself: among the
timestamps in the data, the one that leaves the most users with at least one
interaction on each side (the latest such timestamp, if several tie).

In [8]:
def when(ts: int) -> str:
    return datetime.fromtimestamp(ts, timezone.utc).strftime("%Y-%m-%d %H:%M")


rows = {}
for name in ["timestamp_slicing (fixed)", "timestamp_slicing (best)"]:
    train, test, raw_test = splits[name]
    both = set(train["user_id"]) & set(raw_test["user_id"])
    rows[name] = {
        "pivot": when(raw_test["timestamp"].min() if "best" in name else PIVOT),
        "newest train row": when(train["timestamp"].max()),
        "oldest test row": when(raw_test["timestamp"].min()),
        "users on both sides": len(both),
        "test users unknown to train": raw_test["user_id"].n_unique() - len(both),
    }
pd.DataFrame(rows).T

,pivot,newest train row,oldest test row,users on both sides,test users unknown to train
timestamp_slicing (fixed),1998-03-01 00:00,1998-02-28 22:47,1998-03-01 01:23,120,207
timestamp_slicing (best),1998-01-13 00:48,1998-01-13 00:47,1998-01-13 00:48,141,353


Both pivots split the data cleanly in time. The `best` pivot falls in mid-January
1998, earlier than the fixed one, and keeps more users on both sides (141 against
120), at the cost of a much larger test set: 40,032 rows before alignment
against 22,015. Either way, every user who starts after the pivot is lost to
evaluation.

### Cold start

The cold-start strategies promise that the entities they hold out have no
training row at all:

In [9]:
rows = {}
for name, column in [("item_cold_start", "item_id"), ("user_cold_start", "user_id")]:
    train, test, raw_test = splits[name]
    held_out = raw_test[column].unique().to_list()
    rows[name] = {
        "held out": len(held_out),
        "of": ratings[column].n_unique(),
        "held out with a train row": train[column].is_in(held_out).sum(),
        "held out still in test": test[column].n_unique(),
    }
pd.DataFrame(rows).T

,held out,of,held out with a train row,held out still in test
item_cold_start,168,1682,0,168
user_cold_start,94,943,0,94


## Seeds and reproducibility

The random strategies and the cold-start sample read `seed`. The same seed gives
the same split; another seed gives another one. The temporal strategies and
slicing have nothing random to seed.

A seed reproduces a split **of the same rows in the same order**: the random
draw follows the row order of the frame. The last column runs the same seed on
the same ratings in shuffled order:

In [10]:
def held_out(frame, spec: SplitSpec) -> set:
    _, _, test = splitter.split_transaction(frame, test=spec)
    return set(test.to_native().select("user_id", "item_id").iter_rows())


shuffled = ratings.sample(fraction=1.0, shuffle=True, seed=0)
rows = {}
for strategy, param in [
    ("random_holdout", {"ratio": 0.1}),
    ("random_leave_k_out", {"k": 1}),
    ("item_cold_start", {"ratio": 0.1}),
    ("user_cold_start", {"ratio": 0.1}),
]:
    base = held_out(data, SplitSpec(strategy=strategy, seed=42, **param))
    rows[strategy] = {
        "seed 42 again": base
        == held_out(data, SplitSpec(strategy=strategy, seed=42, **param)),
        "seed 7": base == held_out(data, SplitSpec(strategy=strategy, seed=7, **param)),
        "seed 42, shuffled file": base
        == held_out(shuffled, SplitSpec(strategy=strategy, seed=42, **param)),
    }
pd.DataFrame(rows).T

,seed 42 again,seed 7,"seed 42, shuffled file"
random_holdout,True,False,False
random_leave_k_out,True,False,False
item_cold_start,True,False,True
user_cold_start,True,False,True


The cold-start strategies sample from the sorted list of distinct ids, so they
do not depend on row order. To reproduce any split exactly, keep the input
fixed as well as the seed, or save the split once and load it from then on
(`writer.save_split`, guide 1).

## Adding a validation split

The validation split is made **from the training set the test split left**,
never from the full data, so validation rows can never be test rows. Its
`ratio` is a fraction of that training set.

### A single validation holdout

Here both splits are a 10% temporal holdout. The log shows that the test set is
aligned twice: once against the training set of the test split, and again
against the smaller one the validation split left.

In [11]:
logger.enable("warprec")
train, validation, test = splitter.split_transaction(
    data,
    test=SplitSpec(strategy="temporal_holdout", ratio=0.1),
    validation=SplitSpec(strategy="temporal_holdout", ratio=0.1),
)
logger.disable("warprec")
print(f"train {len(train):,} | validation {len(validation):,} | test {len(test):,}")

08-10-2026 12:22:26 - WarpRec - 📝 Starting test splitting process with temporal_holdout splitting strategy.


08-10-2026 12:22:26 - WarpRec - 📝 Test splitting completed in : 0.01s


08-10-2026 12:22:26 - WarpRec - 📝 Starting validation splitting process with temporal_holdout splitting strategy.


08-10-2026 12:22:26 - WarpRec - 📝 Validation splitting completed in : 0.00s


08-10-2026 12:22:26 - WarpRec - ✅ Splitting process over in 0.01s.


08-10-2026 12:22:26 - WarpRec - 📢 Test set was not aligned with the training set. Filtered out 38 transactions.


08-10-2026 12:22:26 - WarpRec - 📢 Validation set was not aligned with the training set. Filtered out 37 transactions.


08-10-2026 12:22:26 - WarpRec - 📢 Test set was not aligned with the training set. Filtered out 10 transactions.


train 80,187 | validation 9,337 | test 10,391


With a single validation split, the returned `train` is what **both** datasets
train on. The validation dataset trains on it and evaluates on `validation`; the
main dataset, the one the final model is trained on and tested with, trains on
the same rows and evaluates on `test`. The validation rows are never trained on,
even by the final model:

In [12]:
test_only_train = splits["temporal_holdout"][0]
pd.DataFrame(
    [
        {
            "dataset": "main",
            "trains on": len(train),
            "evaluates on": len(test),
            "evaluation set": "test",
        },
        {
            "dataset": "validation",
            "trains on": len(train),
            "evaluates on": len(validation),
            "evaluation set": "validation",
        },
        {
            "dataset": "main, without validation",
            "trains on": len(test_only_train),
            "evaluates on": len(splits["temporal_holdout"][1]),
            "evaluation set": "test",
        },
    ]
).set_index("dataset")

,trains on,evaluates on,evaluation set
dataset,,,
main,80187,10391,test
validation,80187,9337,validation
"main, without validation",89561,10401,test


Adding the validation split took 9,374 rows (9,337 after alignment) out of the
final model's training data. Use k-fold validation when the final model should
train on everything the test split left.

### K-fold validation

`k_fold_cross_validation` deals each user's training rows into `folds` folds in
turn, in file order: the user's first row to fold 1, the second to fold 2, and
so on. There is no shuffle and `seed` is not read, so shuffle the file
beforehand if its order means something. Each fold validates on its own part
and trains on the others. `validation` is now a list of `(train, validation)`
pairs, and the main `train` is the whole training set of the test split:

In [13]:
train, folds, test = splitter.split_transaction(
    data,
    test=SplitSpec(strategy="temporal_holdout", ratio=0.1),
    validation=SplitSpec(strategy="k_fold_cross_validation", folds=3),
)
print(f"main: train {len(train):,} | test {len(test):,}")
pd.DataFrame(
    [
        {"fold": i + 1, "trains on": len(t), "validates on": len(v)}
        for i, (t, v) in enumerate(folds)
    ]
).set_index("fold")

main: train 89,561 | test 10,401


,trains on,validates on
fold,,
1,59434,30054
2,59709,29767
3,59979,29517


Before alignment, the three validation parts are a partition of the training
set: every training row is validated exactly once.

In [14]:
raw_folds = splitter.process_split(
    train, SplitSpec(strategy="k_fold_cross_validation", folds=3)
)
validated = pl.concat([v.to_native() for _, v in raw_folds])
print(
    len(validated) == len(train),
    validated.sort("user_id", "item_id").equals(
        train.to_native().sort("user_id", "item_id")
    ),
)

True True


Each fold's validation set is aligned with that fold's own training set, which is
why the fold rows above sum to a little less than the 89,561 training rows.

## The same splits from a configuration

In a configuration file, the `splitter` section holds `test_splitting` and,
optionally, `validation_splitting`, each with the same keys as a `SplitSpec`.
Unlike the API, the configuration checks that `ratio`, `k`, `timestamp` or
`folds` is set for the holdout, leave-k-out, slicing and k-fold strategies, and
refuses `k_fold_cross_validation` as a test split.

In [15]:
print(Path("configs/leave-one-out.yml").read_text())

# Hold out one random interaction per user as the test set.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: explicit
splitter:
  test_splitting:
    strategy: random_leave_k_out
    k: 1
    seed: 42
models:
  Pop: {}
evaluation:
  top_k: [10]
  metrics: [nDCG]



`initialize_datasets` reads, splits (with alignment) and builds the datasets,
as in guide 1. The training set has the size of the API's `random_leave_k_out`
split above:

In [16]:
from warprec.common.initialize import initialize_datasets
from warprec.data.reader import ReaderFactory
from warprec.utils.callback import WarpRecCallback
from warprec.utils.config.config import load_design_configuration


def datasets(path: str):
    config = load_design_configuration(path)
    return initialize_datasets(
        ReaderFactory.get_reader(config), WarpRecCallback(), config
    )


main, _, _ = datasets("configs/leave-one-out.yml")
print(
    main.train_set.get_sparse().nnz,
    len(splits["random_leave_k_out"][0]),
    "| test:",
    main.eval_set.get_sparse().nnz,
)

99057 99057 | test: 942


The two validation layouts are a `validation_splitting` entry each:

In [17]:
print(Path("configs/validation-holdout.yml").read_text())

# Test: each user's latest 10%. Validation: the latest 10% of what is left.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: explicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
  validation_splitting:
    strategy: temporal_holdout
    ratio: 0.1
models:
  Pop: {}
evaluation:
  top_k: [10]
  metrics: [nDCG]



In [18]:
print(Path("configs/validation-kfold.yml").read_text())

# Test: each user's latest 10%. Validation: 3 folds of what is left.
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: explicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.1
  validation_splitting:
    strategy: k_fold_cross_validation
    folds: 3
models:
  Pop: {}
evaluation:
  top_k: [10]
  metrics: [nDCG]



Run, they give the datasets described above: with the holdout, the main and the
validation dataset train on the same 80,187 rows; with folds, the main dataset
trains on all 89,561 and each fold dataset on its own part.

In [19]:
def nnz(dataset) -> tuple[int, int]:
    return dataset.train_set.get_sparse().nnz, dataset.eval_set.get_sparse().nnz


rows = []
for path in ["configs/validation-holdout.yml", "configs/validation-kfold.yml"]:
    main, validation, fold_datasets = datasets(path)
    named = [("main", main)]
    named += [("validation", validation)] if validation is not None else []
    named += [(f"fold {i + 1}", d) for i, d in enumerate(fold_datasets)]
    for name, dataset in named:
        trains_on, evaluates_on = nnz(dataset)
        rows.append(
            {
                "config": Path(path).stem,
                "dataset": name,
                "trains on": trains_on,
                "evaluates on": evaluates_on,
            }
        )
pd.DataFrame(rows).set_index(["config", "dataset"])

trains on  evaluates on
config             dataset                            
validation-holdout main            80187         10391
                   validation      80187          9337
validation-kfold   main            89561         10401
                   fold 1          59434         30054
                   fold 2          59709         29767
                   fold 3          59979         29517

In the training pipeline the validation dataset (or the folds) drives the
hyperparameter search, and the main dataset trains the final model and reports
its test results; guide 10 covers that.

## Summary of the `splitter` section

| key | what it does |
|---|---|
| `test_splitting.strategy` | required; any strategy except `k_fold_cross_validation` |
| `validation_splitting.strategy` | optional; a holdout, leave-k-out, slicing or k-fold strategy, applied to the training set the test split left |
| `ratio` | fraction of rows (holdouts) or of distinct items/users (cold start) |
| `k` | rows held out per user (leave-k-out); users with `k` or fewer rows are removed |
| `timestamp` | the pivot of `timestamp_slicing`: an integer, or `best` |
| `folds` | the number of folds of `k_fold_cross_validation` |
| `seed` | the random strategies' and cold-start sample's seed, 42 by default |

The [Splitting configuration](https://warprec.readthedocs.io/en/latest/configuration/splitting/)
page lists the strategies; [Cold-Start Protocols](https://warprec.readthedocs.io/en/latest/evaluation/cold-start/)
covers evaluating the cold-start splits.